# 🟡 Optional Extensions — L02 Probability & Statistics

**This notebook is entirely optional.** It is for curious learners who want the deeper
mathematical story behind the concepts taught in Parts 1–3. Nothing in here is assessed
or required for later lessons.

**Topics covered:**
| Section | What you'll explore |
|---|---|
| 1. Bayes' Theorem | The screening-test example — why base rates matter |
| 2. T-test derivation | Where the t-statistic formula comes from |
| 3. Bootstrapping | Estimating uncertainty by resampling your own data |
| 4. CLT proof sketch | Why sample means converge to normal |

**Pre-requisite:** Complete the three Part notebooks first. This notebook builds on concepts
introduced there without re-explaining them.


In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
from scipy.stats import t as t_dist
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Libraries loaded — let's go deeper!")


---
## Section 1 — Bayes' Theorem: Why Base Rates Matter

### The screening-test problem

Imagine a medical screening test for a rare disease. The disease affects 1% of the population.
The test is 95% accurate — if you have the disease, it says "positive" 95% of the time.
If you *don't* have the disease, it says "positive" only 5% of the time (a false alarm).

**You test positive. What is the probability you actually have the disease?**

Most people answer "about 95%." The correct answer is shockingly low.

**Bayes' theorem** is the formula that gets the right answer.


In [ ]:
# Bayes' theorem — the screening test in code
# All numbers computed via simulation to make the logic concrete

np.random.seed(42)
N_POPULATION = 100_000

# Base rate: 1% of the population has the disease
disease_status = np.random.binomial(1, 0.01, size=N_POPULATION)   # 1 = has disease
n_with_disease    = disease_status.sum()
n_without_disease = N_POPULATION - n_with_disease

# Test sensitivity: 95% chance of a positive test if you have the disease
# Test specificity: 5% chance of a positive test if you DON'T have the disease (false positive rate)
test_sensitivity  = 0.95   # P(positive | disease)
false_positive_rate = 0.05   # P(positive | no disease)

# Simulate test results
test_positive = np.zeros(N_POPULATION, dtype=int)
test_positive[disease_status == 1] = np.random.binomial(1, test_sensitivity, n_with_disease)
test_positive[disease_status == 0] = np.random.binomial(1, false_positive_rate, n_without_disease)

# Key numbers
true_positives  = ((disease_status == 1) & (test_positive == 1)).sum()
false_positives = ((disease_status == 0) & (test_positive == 1)).sum()
total_positives = test_positive.sum()

print(f"=== Screening Test Results (N = {N_POPULATION:,}) ===")
print()
print(f"People with the disease:    {n_with_disease:,}  ({n_with_disease/N_POPULATION:.1%})")
print(f"People without the disease: {n_without_disease:,}")
print()
print(f"True positives:  {true_positives:,}   (have disease AND tested positive)")
print(f"False positives: {false_positives:,}  (no disease BUT tested positive)")
print(f"Total positives: {total_positives:,}")
print()

# The key result: P(disease | positive test)
p_disease_given_positive = true_positives / total_positives
print(f"P(disease | positive test) = {true_positives}/{total_positives} = {p_disease_given_positive:.1%}")
print()
print("This is Bayes' theorem in action:")
print(f"  Even with a 95% accurate test, a positive result only means")
print(f"  {p_disease_given_positive:.0%} probability of actually having the disease.")
print(f"  Why? Because {false_positives:,} people who don't have the disease")
print(f"  still tested positive — swamping the {true_positives:,} true positives.")


In [ ]:
# Visualise the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))

conf_matrix = np.array([
    [true_positives, false_positives],
    [((disease_status==1) & (test_positive==0)).sum(),   # true negatives (missed)
     ((disease_status==0) & (test_positive==0)).sum()]   # true negatives (correct)
])

im = ax.imshow(conf_matrix, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Has Disease', 'No Disease'], fontsize=12)
ax.set_yticklabels(['Positive Test', 'Negative Test'], fontsize=12)

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{conf_matrix[i, j]:,}', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if conf_matrix[i, j] > conf_matrix.max()/2 else 'black')

ax.set_title(f"Confusion Matrix — Screening Test\n"
             f"Sensitivity 95%, False Positive Rate 5%, Disease Prevalence 1%",
             fontsize=12)
ax.set_xlabel("True Status", fontsize=12)
ax.set_ylabel("Test Result", fontsize=12)
plt.colorbar(im)
plt.tight_layout()
plt.show()

print("The top-left cell (true positives) is tiny compared to the top-right cell (false positives).")
print("That's why a positive test has a much lower predictive value than people expect.")


### The Bayes' Theorem formula

The analytical version of what we just computed by simulation:

```
P(disease | positive) = P(positive | disease) × P(disease)
                        ─────────────────────────────────────
                               P(positive)

where:
  P(positive) = P(pos | disease) × P(disease) + P(pos | no disease) × P(no disease)
              = 0.95 × 0.01  +  0.05 × 0.99
              = 0.0095 + 0.0495
              = 0.059
```

So: P(disease | positive) = (0.95 × 0.01) / 0.059 ≈ **16.1%**

### Quick Check

**Q:** Suppose the disease affects 10% of the population instead of 1%. Same test (95% sensitivity, 5% FPR). Now what is P(disease | positive)?

> **Sample answer:** P(positive) = 0.95 × 0.10 + 0.05 × 0.90 = 0.095 + 0.045 = 0.14
> P(disease | positive) = 0.095 / 0.14 ≈ 67.9%.
> The base rate (10% vs 1%) makes a huge difference — the same positive test now means 68% probability rather than 16%. This is why Bayes' theorem is central to medical screening policy: the higher the prevalence in the screened population, the more trustworthy a positive test becomes.


---
## Section 2 — Where the T-statistic Comes From

The two-proportion z-test we used in Part 3 works when sample sizes are large (n ≥ 30).
For smaller samples — or when comparing *means* (not just proportions) — we use the **t-test**.

### The t-statistic intuition

The t-statistic is a standardised measure of how different two group means are, given the variability within each group.

For a two-sample comparison:
```
t = (x̄₁ − x̄₂) / sqrt(s₁²/n₁ + s₂²/n₂)
```
where x̄ is the sample mean, s² is the sample variance, and n is the sample size.

The denominator is the **standard error of the difference** — how much we'd expect the difference to vary by chance alone. A large t-statistic means the difference is big relative to the natural variability.

The t-statistic is then compared to a **t-distribution** (which has heavier tails than the normal distribution for small samples) to get a p-value.


In [ ]:
# T-test: comparing onboarding times from the assignment data
np.random.seed(42)
N_OLD = 500
N_NEW = 500
old_onboard_minutes = np.random.normal(loc=22, scale=6, size=N_OLD).clip(5, 60)
new_onboard_minutes = np.random.normal(loc=11, scale=3, size=N_NEW).clip(3, 30)

# Manual t-statistic computation
x1, x2 = old_onboard_minutes.mean(), new_onboard_minutes.mean()
s1, s2 = old_onboard_minutes.var(ddof=1), new_onboard_minutes.var(ddof=1)
n1, n2 = len(old_onboard_minutes), len(new_onboard_minutes)

se_diff = np.sqrt(s1/n1 + s2/n2)
t_manual = (x1 - x2) / se_diff

# Compare to scipy
t_scipy, p_scipy = st.ttest_ind(old_onboard_minutes, new_onboard_minutes)

print("=== T-test: Old vs New Onboarding Time ===")
print()
print(f"Old flow mean: {x1:.2f} min  |  New flow mean: {x2:.2f} min")
print(f"Difference:    {x1 - x2:.2f} min")
print()
print(f"SE of difference (denominator): {se_diff:.4f}")
print(f"T-statistic (manual):           {t_manual:.4f}")
print(f"T-statistic (scipy):            {t_scipy:.4f}")
print(f"P-value (scipy):                {p_scipy:.6f}")
print()

# Visualise the t-distribution and our statistic
df = n1 + n2 - 2   # degrees of freedom
x_range = np.linspace(-50, 50, 500)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_range, t_dist.pdf(x_range, df=df), 'b-', linewidth=2, label=f't-distribution (df={df})')
ax.axvline(t_scipy, color='red', linewidth=2.5, linestyle='--',
           label=f't-statistic = {t_scipy:.2f}')
ax.axvline(-abs(t_scipy), color='red', linewidth=2.5, linestyle='--')
# Shade the tails
x_tail_right = np.linspace(abs(t_scipy), 50, 200)
x_tail_left  = np.linspace(-50, -abs(t_scipy), 200)
ax.fill_between(x_tail_right, t_dist.pdf(x_tail_right, df=df), alpha=0.3, color='red', label='p-value area')
ax.fill_between(x_tail_left,  t_dist.pdf(x_tail_left,  df=df), alpha=0.3, color='red')
ax.set_xlabel("T-statistic")
ax.set_ylabel("Density")
ax.set_title("T-distribution: Our Test Statistic vs the Null Distribution")
ax.set_xlim(-60, 60)
ax.legend()
plt.tight_layout()
plt.show()

print("The red shaded area (the tails beyond our t-statistic) is the p-value.")
print("With such a large t-statistic, the p-value is essentially zero.")


### Quick Check

**Q:** If the t-statistic were 1.8 instead of the large value above, would you expect a higher or lower p-value? Why?

> **Sample answer:** Higher. A smaller t-statistic means the difference between groups is smaller relative to the within-group variability. The red shaded tail area would be larger (covering more of the distribution), giving a higher p-value — meaning the result is less unusual under the null hypothesis.


---
## Section 3 — Bootstrapping: Estimating Uncertainty by Resampling

Bootstrapping is a powerful technique for estimating confidence intervals without
knowing the formula. It works by:

1. Taking your sample data (size n)
2. Drawing n random values *from the sample itself* (with replacement)
3. Computing your statistic on this "bootstrap sample"
4. Repeating 1,000+ times
5. Using the spread of results as your uncertainty estimate

It works because your sample is the best available model of the population.
Drawing from it simulates what you'd get from new samples from the population.


In [ ]:
# Bootstrap CI for the median onboarding time (old flow)
# The median has no simple closed-form CI — this is where bootstrapping shines

np.random.seed(42)
data = old_onboard_minutes.copy()
n    = len(data)

N_BOOTSTRAP = 5000
bootstrap_medians = []

for _ in range(N_BOOTSTRAP):
    # Draw n values from the data WITH replacement
    bootstrap_sample = np.random.choice(data, size=n, replace=True)
    bootstrap_medians.append(np.median(bootstrap_sample))

bootstrap_medians = np.array(bootstrap_medians)

# 95% CI: use the 2.5th and 97.5th percentiles of the bootstrap distribution
ci_low  = np.percentile(bootstrap_medians, 2.5)
ci_high = np.percentile(bootstrap_medians, 97.5)

print(f"=== Bootstrap CI for Median Onboarding Time ===")
print(f"Observed median: {np.median(data):.1f} minutes")
print(f"Bootstrap 95% CI: [{ci_low:.1f}, {ci_high:.1f}] minutes")
print(f"CI width: {ci_high - ci_low:.1f} minutes")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(bootstrap_medians, bins=50, color='mediumpurple', edgecolor='white', alpha=0.85)
ax.axvline(np.median(data), color='orange', linewidth=2.5, label=f'Observed median: {np.median(data):.1f}')
ax.axvline(ci_low,  color='green', linewidth=2.5, linestyle='--', label=f'95% CI: [{ci_low:.1f}, {ci_high:.1f}]')
ax.axvline(ci_high, color='green', linewidth=2.5, linestyle='--')
ax.axvspan(ci_low, ci_high, alpha=0.15, color='green')
ax.set_xlabel("Bootstrap median onboarding time (minutes)")
ax.set_ylabel("Count")
ax.set_title(f"Bootstrap Distribution of Median\n({N_BOOTSTRAP:,} resamples)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nBootstrapping is formula-free — it works for any statistic:")
print("mean, median, correlation coefficient, model accuracy, anything.")


---
## Section 4 — The CLT: Why It Works (Proof Sketch)

For the mathematically curious, here is an intuitive sketch of why sample means become normal.

### The moment-generating function argument

The key insight is that the *moment-generating function* (MGF) of a sum of independent random variables is the **product** of their individual MGFs. For a large sum, this product converges to the MGF of the normal distribution — a result from real analysis.

We won't prove this formally, but we can see the consequence clearly.

**The 3Blue1Brown connection:** The best visual proof of the CLT uses the concept of *convolutions* — the mathematical operation behind adding random variables. The 3Blue1Brown video on the CLT ([link](https://www.youtube.com/watch?v=zeJD6dqJ5lo)) makes this strikingly clear.

### Code illustration: from uniform to normal

We already saw this in `03_confidence_intervals.ipynb` with dice. Here we extend it
to show how *any* distribution — even a highly irregular one — converges.


In [ ]:
# CLT: From bimodal to normal
# A bimodal distribution (two peaks) is maximally non-normal
# Show that sample means still converge

np.random.seed(42)

# Bimodal source: mix of two normals (like two distinct customer segments)
def bimodal_sample(n):
    group = np.random.binomial(1, 0.5, n)
    result = np.where(group == 0,
                      np.random.normal(2, 0.5, n),
                      np.random.normal(8, 0.5, n))
    return result

n_experiments = 3000

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

sample_sizes = [1, 5, 20, 100]
for ax, ss in zip(axes, sample_sizes):
    means = [bimodal_sample(ss).mean() for _ in range(n_experiments)]
    ax.hist(means, bins=50, color='teal', edgecolor='white', alpha=0.85, density=True)
    ax.set_title(f'n = {ss}\n(average of {ss} draws)')
    ax.set_xlabel('Sample mean')
    ax.set_ylabel('Density' if ax == axes[0] else '')

axes[0].set_title("n = 1 (raw bimodal\nsource)")
plt.suptitle("CLT in Action: Bimodal Source → Normal Sample Means",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The source distribution has TWO peaks — completely non-normal.")
print("By n=20 the distribution of sample means already looks approximately normal.")
print("By n=100 it is very close to a perfect bell curve.")
print()
print("This is the CLT. No matter how weird the source, means converge to normal.")
print("Watch 3Blue1Brown's CLT video for the beautiful visual proof:")
print("  https://www.youtube.com/watch?v=zeJD6dqJ5lo")


---

## 🏁 Optional Extensions Complete

You've covered four topics that sit beneath the surface of every statistical test
used in machine learning:

| Topic | Key insight |
|---|---|
| **Bayes' Theorem** | Base rates dominate. A 95% accurate test can still be mostly wrong if the disease is rare. |
| **T-test derivation** | The t-statistic measures how large a difference is relative to within-group noise. |
| **Bootstrapping** | You can estimate uncertainty for any statistic by resampling from your own data. |
| **CLT proof sketch** | Sample means converge to normal because addition in probability space is convolution — and convolutions converge. |

None of this is assessed. All of it will make you a better practitioner.

**Next:** Return to `assignment.ipynb` if you haven't finished it, or move on to `L03 — Supervised Learning`.
